In [1]:
import pandas as pd

In [2]:
df  = pd.read_csv("../data/cleaned/msg-reply-pairs.csv")

In [3]:
df.head(10)

,contact_person,input_text,reply_text
0,+92 318 0067351,aoa yaar main na tumhare sath DE repeat kr rha...,Sahi hy.. Main university pounch kar tum sy ly...
1,+92 318 0067351,Ok M 8 30.pr niklu ga yha se agr tune phle Jan...,Mera class fellow same hostel main hy room b6 ...
2,+92 318 0067351,Assignment de di thi ?,Han
3,+92 318 0067351,Assalamualaikum yaar mene assignment submit ka...,Wo tu submit hogai.. Sir ly gay
4,Ya bta de m de jata hu 2,30 bje,Office chalay ja
5,+92 318 0067351,This message was deleted Kaha pr h office?,Yaar.. Main abhi class main hoon Office nahi pata
6,+92 318 0067351,This message was deleted,Ruk pouhta hoon Cr pata nahi kidr Class nahi aya
7,+92 318 0067351,Kidr claas m hou ?,Nahi Main ghar hoon Kal aho ga o
8,+92 318 0067351,This message was deleted This message was dele...,Sorry jani.. Abhi message seen kiya hy Tera 🥲
9,+92 318 0067351,Aaj h quiz ya kl h?,aaj hy


In [4]:
df = df.dropna()

In [5]:
import string

def tokenize(text):
    if not isinstance(text, str):
        # handle NaN/float by converting to empty string
        try:
            if pd.isna(text):
                text = ""
            else:
                text = str(text)
        except Exception:
            text = ""
    translator = str.maketrans('', '', string.punctuation)
    return text.lower().translate(translator).split()

In [6]:
vocab = {"<UNK>" : 0}

In [7]:
def build_vocab(row):
   merged_token = tokenize(row["input_text"]) + tokenize(row["reply_text"])

   for token in merged_token:
      if token not in vocab:
         vocab[token] = len(vocab)

In [8]:
df.apply(build_vocab, axis=1)

0       None
1       None
2       None
3       None
4       None
        ... 
7291    None
7292    None
7293    None
7294    None
7295    None
Length: 7295, dtype: object

In [9]:
len(vocab)

10784

In [10]:
# text to indices
def text_to_indices(text):
    indices = []

    for token in tokenize(text):
        if token in vocab:
            indices.append(vocab[token])
        else:
            indices.append(vocab['<UNK>'])

    return indices

In [11]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

# select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
# enable cuDNN autotuner for some speedups (optional)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

use_pin_memory = torch.cuda.is_available()

Using device: cuda


In [12]:
class customDataset(Dataset):
    def __init__(self, df, vocab):
        super().__init__()
        self.df = df
        self.vocab = vocab

    def __len__(self):
        return self.df.shape[0]
    
    def __getitem__(self, index):
        x = text_to_indices(self.df.iloc[index]['input_text'])
        y = text_to_indices(self.df.iloc[index]['reply_text'])
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

In [13]:
dataset = customDataset(df, vocab)

In [14]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True, pin_memory=use_pin_memory)

In [15]:
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=50, hidden_size=64):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_text):
        if input_text.dim() == 1:
            input_text = input_text.unsqueeze(0)  # make batch dimension

        emb = self.embedding(input_text)  # (batch, seq_len, emb_dim)
        out, h_n = self.rnn(emb)  # out: (batch, seq_len, hidden_size)
        logits = self.fc(out)  # (batch, seq_len, vocab_size)
        return logits

In [16]:
learning_rate = 0.001
epochs = 20

In [17]:
model = SimpleRNN(len(vocab)).to(device)

In [18]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [19]:
total_loss = 0
for epoch in range(epochs):
    total_loss = 0
    model.train()
    for msg, reply in dataloader:

        # move tensors to device early
        msg = msg.to(device, non_blocking=True)
        reply = reply.to(device, non_blocking=True)

        if msg.dim() == 1:
            msg = msg.unsqueeze(0)
        if reply.dim() == 1:
            reply = reply.unsqueeze(0)

        # skip empty sequences
        if msg.size(1) == 0 or reply.size(1) == 0:
            continue

        output = model(msg)  # (batch, seq_len_in, vocab_size)

        # Truncate to the minimum sequence length between input and target
        min_len = min(output.size(1), reply.size(1))
        if min_len == 0:
            continue
        logits = output[:, :min_len, :].contiguous().view(-1, output.size(-1))  # (batch*min_len, vocab_size)
        targets = reply[:, :min_len].contiguous().view(-1)  # (batch*min_len,)

        optimizer.zero_grad()
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    
    print(f"{epoch} : {total_loss}")

0 : 50132.44892799854
1 : 43914.55085000396
2 : 41093.919424280524
3 : 38805.507498413324
4 : 36969.48102683574
5 : 35351.52362381667
6 : 33983.2851068154
7 : 32714.104175131768
8 : 31530.68014767021
9 : 30623.10983076319
10 : 29655.420193959028
11 : 28848.286698195152
12 : 28172.586154274642
13 : 27501.26494542323
14 : 26826.11674986873
15 : 26288.72545935586
16 : 25930.331403299235
17 : 25177.11099002324
18 : 24940.505514978897
19 : 24592.57074566651


In [29]:
inv_vocab = {index: token for token, index in vocab.items()}


def predict_reply(message, max_tokens=None):
    tokens = text_to_indices(message)
    if not tokens:
        return ""

    input_tensor = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0)

    model.eval()
    with torch.no_grad():
        logits = model(input_tensor)
        predicted_ids = logits.argmax(dim=-1).squeeze(0).tolist()

    if max_tokens is not None:
        predicted_ids = predicted_ids[:max_tokens]

    predicted_tokens = [inv_vocab.get(token_id, "<UNK>") for token_id in predicted_ids]
    predicted_tokens = [token for token in predicted_tokens if token != "<UNK>"]
    return " ".join(predicted_tokens)


print(predict_reply("kia hal hy?"))

kidr baat hy
